# Regression Tree from Scratch

The aim of this notebook is to implement the **regression tree** algorithm from scratch using only NumPy.

In [45]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load the diabetes dataset
df = pd.DataFrame(load_diabetes().data, columns=load_diabetes().feature_names)
df['target'] = load_diabetes().target # diabete progression measure after one year

In [3]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [4]:
print("Shape:\n", df.shape)
print("Head:\n", df.head())
print("Description:\n", df.describe())
print("Null Values:\n", df.isnull().sum())

Shape:
 (442, 11)
Head:
         age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  
Description:
                 age           sex           bmi            bp            s1  \
count  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02  4.420000e+02   
mean  -2.511817e-19  1.230790e-17 -2.245564e-16 -4.797570e-17 -1.381499e-17   
std    4.761905e-02  

We notice:

- **442 examples, 11 features**: age, sex, bmi, bp, s1, s2, s3, s4, s5, s6 and target.
- **No missing/null values**.
- When looking at the std line in the df.describe() output, we notice that all the features have a standard deviation of 4.76e-02, which is about 0.0476. And the means are all close to zero, meaning the **dataset is already normalized**. Sklearn did it already, so there’s no need for additional preprocessing on the features. 

In [6]:
# Prepare the data: target variable and features
X = df.drop("target", axis=1).values # features
y = df["target"].values # target

In [24]:
# Train and test datasets
n_examples = len(X[:, 0]) # number of examples in the dataset
random_indices = np.random.permutation(n_examples)

X_shuffled = X[random_indices, :]
y_shuffled = y[random_indices]

split = int(n_examples * 0.8)

X_train = X_shuffled[:split, :]
y_train = y_shuffled[:split]
X_test = X_shuffled[split:, :]
y_test = y_shuffled[split:]

In [27]:
# Mean Squared Error function
def compute_mse(y):
    n_examples = y.size
    return (1/n_examples) * np.sum((y - np.mean(y)) ** 2)

In [29]:
# Compute threshold candidatures (midpoints between sorted unique values)
def compute_midpoints(X_feature):
    thresholds = np.unique(X_feature)[:-1] + np.diff(np.unique(X_feature)) / 2
    return thresholds

In [ ]:
# Compute the weighted MSE for a given feature and threshold (to evaluate the quality of a split)
def compute_split_mse(X_feature, y, threshold):
    left_indices = X_feature <= threshold
    right_indices = X_feature > threshold
    
    # If the split is empty, return +infinity (because we want to minimize the weighted MSE)
    if np.sum(left_indices) == 0 or np.sum(right_indices) == 0:
        return np.inf
    
    mse_left = compute_mse(y[left_indices])
    mse_right = compute_mse(y[right_indices])
    
    weighted_mse = (np.sum(left_indices) * mse_left + np.sum(right_indices) * mse_right) / len(y)
    
    return weighted_mse

In [31]:
# Find the best split among all features and thresholds
def best_split(X, y):
    best_gain = np.inf
    best_feature = None
    best_thresh = None

    n_features = X.shape[1]

    # For each feature, find the best threshold and gain
    for feature in range(n_features):
        X_feature = X[:, feature]
        thresholds = compute_midpoints(X_feature)

        for threshold in thresholds:
            gain = compute_split_mse(X_feature, y, threshold)

            if gain < best_gain:
                best_gain = gain
                best_feature = feature
                best_thresh = threshold

    return (best_feature, best_thresh)

In [49]:
# Recursive function to build the regression tree
def build_tree(X, y, depth=0, max_depth=3, min_samples_split=20):
    # Stopping criteria
    if depth >= max_depth or len(y) < min_samples_split:
        return {"leaf": True, "value": np.mean(y)} # return the mean of the target values as the prediction for this leaf node

    feature, threshold = best_split(X, y)

    if feature is None:
        return {"leaf": True, "value": np.mean(y)}

    left_indices = X[:, feature] <= threshold
    right_indices = X[:, feature] > threshold

    # If the split is empty, return the mean of the target values
    if np.sum(left_indices) == 0 or np.sum(right_indices) == 0:
        return {"leaf": True, "value": np.mean(y)}

    # Recursively build the left and right subtrees
    left_subtree = build_tree(X[left_indices], y[left_indices], depth + 1, max_depth, min_samples_split)
    right_subtree = build_tree(X[right_indices], y[right_indices], depth + 1, max_depth, min_samples_split)

    return {"leaf": False, "feature": feature, "threshold": threshold, "left": left_subtree, "right": right_subtree}

**build_tree** constructs the tree by **recursively** partitioning the data. 

Here is the step-by-step process:

**Initial call**: You pass the entire X_train and y_train to depth=0.

**On each call**:

1. **Do we stop?**  If depth >= max_depth or there aren’t enough examples → we return a leaf node with np.mean(y). This is the optimal prediction for these examples.

2. **Otherwise** : we find the best split using best_split(X, y) — the feature and threshold that minimize the weighted MSE.

3. **We partition**: examples with X[:, feature] <= threshold go to the left, the others to the right.

4. **We call build_tree** on each partition with depth + 1. This is where recursion comes in, each subtree is built in the same way.

Visually:

```
build_tree(X, y, depth=0)
├── build_tree(X_left, y_left, depth=1)
│   ├── build_tree(X_left_left, y_left_left, depth=2) → leaf
│   └── build_tree(X_left_right, y_left_right, depth=2) → leaf
└── build_tree(X_right, y_right, depth=1)
    ├── build_tree(X_right_left, y_right_left, depth=2) → leaf
    └── build_tree(X_right_right, y_right_right, depth=2) → leaf
```

In [50]:
# Iterate through the tree for a single example x (a 1D vector)
def predict_one(tree, x):
    # Checks if the tree is a leaf node (True) and returns the prediction
    if tree["leaf"]:
        return tree["value"]
    
    # Unpack the tree node
    feature = tree["feature"]
    threshold = tree["threshold"]
    left_subtree = tree["left"]
    right_subtree = tree["right"]

    # For each sample, traverse the tree according to the feature and threshold values until reaching a leaf node
    if x[feature] <= threshold:
        return predict_one(left_subtree, x)
    else:
        return predict_one(right_subtree, x)

In [51]:
# Prediction function (iterates through the tree for each example in X)
def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])

In [52]:
# Train the tree on the training data
tree = build_tree(X_train, y_train)

In [53]:
# Predict on the test set
y_pred = predict(tree, X_test)

In [54]:
print("Train MSE:", mean_squared_error(y_train, predict(tree, X_train)))
print("Train R2 score:", r2_score(y_train, predict(tree, X_train)))

Train MSE: 2854.555608945099
Train R2 score: 0.5012675119858447


In [55]:
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R2 score:", r2_score(y_test, y_pred))

Test MSE: 3665.9613129469385
Test R2 score: 0.45534228712073355


**Analysis:**

| Configuration                          | Train R² | Test R² | Gap  |
|----------------------------------------|----------|---------|------|
| max_depth=5, min_samples_split=10      | 0.65     | 0.45    | 0.20 |
| max_depth=3, min_samples_split=20      | 0.50     | 0.46    | 0.04 |

**The reasoning:**

A deep tree with few constraints creates many highly specific nodes that fit the details of the training set: it memorizes rather than learns. 

This is **overfitting**: excellent performance on the training set, poor performance on the test set.

By reducing `max_depth` from 5 to 3, we limit the number of successive splits. The tree remains more general.

By increasing `min_samples_split` from 10 to 20, we prevent splits on nodes with too few samples. These nodes become leaves earlier, which simplifies the tree.

**The fundamental trade-off:**

- Too simple → underfitting: Train R² and Test R² are both low and close  
- Too complex → overfitting: high Train R², low Test R², large gap  
- Well-tuned → Train R² and Test R² are close and reasonably high  

The goal is not to maximize Train R², but to minimize the gap between train and test while maintaining an acceptable Test R².